# How the middle layers find out

MichAl Academy, unit 3.3.

Run each cell with **Shift+Enter**.

Lesson 1.12.2 said the slope tells you which way is downhill. This notebook works
out how you get that slope for a weight buried three layers from the answer,
first the obvious way, then the way everybody actually uses, and measures why
the second one won.


In [ ]:
import time

import numpy as np
import torch

# Same reason as unit 3.1: these networks are tiny, and coordinating threads
# costs more than the arithmetic saves.
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)

print("torch", torch.__version__)


## 1. One small network, forward

Two inputs, two hidden units, one output. Nine parameters in total, which is few
enough to check every one of them by hand.


In [ ]:
x = np.array([1.0, 0.5])
W1 = np.array([[0.3, -0.2], [0.4, 0.1]])   # one row per hidden unit
b1 = np.array([0.1, -0.1])
W2 = np.array([0.5, -0.6])
b2 = 0.2
target = 1.0

def sig(v):
    return 1.0 / (1.0 + np.exp(-v))

z1 = W1 @ x + b1
a1 = sig(z1)
z2 = W2 @ a1 + b2
a2 = sig(z2)
loss = (a2 - target) ** 2

print(f"z1   = {z1}")
print(f"a1   = {a1}")
print(f"z2   = {z2:.6f}")
print(f"a2   = {a2:.6f}")
print(f"loss = {loss:.6f}")


The loss exists only at the end. `W1` never saw it.

## 2. Backward, by hand

Work leftwards. Each step multiplies by one local derivative.

The number to watch is `delta2`. It gets computed once and then does three jobs.


In [ ]:
dL_da2 = 2 * (a2 - target)
da2_dz2 = a2 * (1 - a2)
delta2 = dL_da2 * da2_dz2

dL_dW2 = delta2 * a1
dL_db2 = delta2

delta1 = delta2 * W2 * (a1 * (1 - a1))

dL_dW1 = np.outer(delta1, x)
dL_db1 = delta1

print(f"dL/da2 = {dL_da2:.6f}")
print(f"delta2 = {delta2:.6f}   <- computed once")
print()
print(f"dL/dW2 = {dL_dW2}        <- delta2 x a1")
print(f"dL/db2 = {dL_db2:.6f}        <- delta2 itself")
print(f"delta1 = {delta1}        <- built from delta2")
print(f"dL/dW1 =\n{dL_dW1}")
print(f"dL/db1 = {dL_db1}")


Three uses of one number. The bias gradient **is** `delta2`, because a bias is
added with a coefficient of 1. Each second-layer weight gradient is `delta2`
times the activation that weight multiplied. And the first layer's deltas start
from `delta2` rather than from the loss again.

That reuse is the whole of backpropagation.

## 3. Ask autograd for the same nine numbers


In [ ]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)

ta1 = torch.sigmoid(tW1 @ torch.tensor(x) + tb1)
ta2 = torch.sigmoid(tW2 @ ta1 + tb2)
((ta2 - target) ** 2).backward()

print(f"autograd dL/dW2 = {tW2.grad.numpy()}")
print(f"autograd dL/db2 = {tb2.grad.item():.6f}")
print(f"autograd dL/dW1 =\n{tW1.grad.numpy()}")
print(f"autograd dL/db1 = {tb1.grad.numpy()}")

worst = max(
    np.abs(dL_dW2 - tW2.grad.numpy()).max(),
    abs(dL_db2 - tb2.grad.item()),
    np.abs(dL_dW1 - tW1.grad.numpy()).max(),
    np.abs(dL_db1 - tb1.grad.numpy()).max(),
)
print(f"\nlargest disagreement across all nine parameters: {worst:.3e}")


## 4. A gradient really is just a slope

You do not have to take the chain rule on trust. Nudge one weight up and down,
see how the loss moves, divide by the distance. That is a derivative from its
definition.


In [ ]:
def loss_with(W1_):
    return (sig(W2 @ sig(W1_ @ x + b1) + b2) - target) ** 2

print(f"{'nudge':<14}{'slope measured':<18}{'by hand':<18}off by")
for eps in (1e-1, 1e-2, 1e-3, 1e-5, 1e-7):
    up, down = W1.copy(), W1.copy()
    up[0, 0] += eps
    down[0, 0] -= eps
    slope = (loss_with(up) - loss_with(down)) / (2 * eps)
    print(f"{eps:<14.0e}{slope:<18.9f}{dL_dW1[0, 0]:<18.9f}{abs(slope - dL_dW1[0, 0]):.3e}")


It converges, so nudging genuinely measures the gradient.

But look at the last row. The smallest nudge is **not** the most accurate. At
1e-7 the two losses being subtracted agree in so many leading digits that the
subtraction throws away the very difference being measured. Finite differences
have a sweet spot and then get worse.

## 5. So why not just nudge everything?

Because one nudge measures one parameter, and you need two of them per
parameter to get a centred estimate.


In [ ]:
net = torch.nn.Sequential(
    torch.nn.Linear(64, 32), torch.nn.ReLU(),
    torch.nn.Linear(32, 10))

n_params = sum(p.numel() for p in net.parameters())
X = torch.randn(256, 64)
y = torch.randint(0, 10, (256,))
lossf = torch.nn.CrossEntropyLoss()

REP = 200
start = time.time()
for _ in range(REP):
    with torch.no_grad():
        lossf(net(X), y)
forward = (time.time() - start) / REP

start = time.time()
for _ in range(REP):
    net.zero_grad()
    lossf(net(X), y).backward()
both = (time.time() - start) / REP

print(f"network 64-32-10, {n_params} parameters, batch of 256")
print(f"one forward pass              : {forward * 1e3:.4f} ms")
print(f"one forward and one backward  : {both * 1e3:.4f} ms")
print()
print(f"nudging needs {2 * n_params} forward passes: {2 * n_params * forward:.3f} s per step")
print(f"backpropagation                        : {both:.4f} s per step")
print(f"ratio: {2 * n_params * forward / both:.0f}x")


Your numbers will differ from the lesson's, because timings depend on the
machine. The ratio is the part that travels.

Note also that the backward pass is not free but it is cheap: roughly two to
three times a forward pass, for every gradient at once.

## 6. What actually arrives at the front

Each layer the error passes through multiplies it by that layer's local
derivative. So the question is how big those multipliers can get.


In [ ]:
z = torch.linspace(-8, 8, 100001, requires_grad=True)

for name, fn in (("sigmoid", torch.sigmoid), ("tanh", torch.tanh), ("relu", torch.relu)):
    grad, = torch.autograd.grad(fn(z).sum(), z)
    print(f"{name:<9} largest derivative anywhere: {grad.max().item():.4f}")

print(f"\ntwelve sigmoid layers, best case: 0.25 ** 12 = {0.25 ** 12:.3e}")


A sigmoid can never pass back more than a quarter of what reached it. Stack
twelve and the best case is already 6e-08, before the weights have their say.

Measure it on real stacks.


In [ ]:
WIDTH, SEEDS = 16, 15

def first_layer_grad(depth, act, seed, he=False):
    torch.manual_seed(seed)
    layers = []
    for _ in range(depth):
        layers += [torch.nn.Linear(WIDTH, WIDTH), act()]
    layers += [torch.nn.Linear(WIDTH, 1)]
    net = torch.nn.Sequential(*layers)
    if he:
        for m in net:
            if isinstance(m, torch.nn.Linear):
                torch.nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                torch.nn.init.zeros_(m.bias)
    Xd, yd = torch.randn(64, WIDTH), torch.randn(64, 1)
    net.zero_grad()
    ((net(Xd) - yd) ** 2).mean().backward()
    return [m.weight.grad.norm().item() for m in net if isinstance(m, torch.nn.Linear)][0]

ACTS = (("sigmoid", torch.nn.Sigmoid), ("tanh", torch.nn.Tanh), ("relu", torch.nn.ReLU))
DEPTHS = (4, 8, 12, 16)

print("median gradient at the FIRST layer, over", SEEDS, "random starts\n")
print(f"{'act':<9}" + "".join(f"{d:>2} layers  " for d in DEPTHS))
for name, act in ACTS:
    cells = "".join(
        f"{np.median([first_layer_grad(d, act, s) for s in range(SEEDS)]):>10.3e}"
        for d in DEPTHS)
    print(f"{name:<9}{cells}")


Sixteen sigmoid layers leave the first layer with a gradient around 2e-14. That
layer is not learning. This is the vanishing gradient problem.

Now read the ReLU row against the tanh row. ReLU is supposed to be the fix, and
it is losing.


In [ ]:
wins = 0
for depth in DEPTHS:
    for seed in range(SEEDS):
        t = first_layer_grad(depth, torch.nn.Tanh, seed)
        r = first_layer_grad(depth, torch.nn.ReLU, seed)
        if r > t:
            wins += 1

total = len(DEPTHS) * SEEDS
print(f"relu beat tanh at the first layer in {wins} of {total} runs")


## 7. The variable that was missing

Those runs used whatever PyTorch does by default. Read what that actually is.


In [ ]:
import inspect

print(inspect.getsource(torch.nn.Linear.reset_parameters))


`kaiming_uniform_` with `a=sqrt(5)`, which the comment in that source says is
the same as uniform between plus and minus 1 over the square root of the input
count. That is a sensible general default. It is not the scheme He and
colleagues designed for ReLU in 2015.

Give ReLU the initialisation it was designed for.


In [ ]:
for depth in DEPTHS:
    default = np.median([first_layer_grad(depth, torch.nn.ReLU, s) for s in range(SEEDS)])
    he = np.median([first_layer_grad(depth, torch.nn.ReLU, s, he=True) for s in range(SEEDS)])
    print(f"depth {depth:>2}:  default {default:.3e}    He {he:.3e}")

he_wins = sum(
    1 for s in range(SEEDS)
    if first_layer_grad(16, torch.nn.ReLU, s, he=True) > first_layer_grad(16, torch.nn.Tanh, s))
print(f"\nat 16 layers, relu+He beats tanh+default in {he_wins} of {SEEDS} runs")


## 8. What you have

- A weight in the middle learns because the error is carried back to it, one
  local derivative at a time.
- Nudging measures the same gradient correctly and is unaffordable: one
  parameter per nudge against every gradient in one backward pass.
- The saving is reuse. `delta2` is computed once and serves the whole layer and
  the layer before it.
- Gradients change size as they travel, and sigmoid cannot pass back more than
  a quarter of what it received.
- An activation function has no gradient behaviour on its own. It has one in
  combination with an initialisation, and swapping the initialisation reversed
  which activation looked better.

Unit 3.5 is the rest of the settings that decide whether training converges.
